# Chapter 1
In this chapter we look at how to retrieve, read, visualise, extract important details, crop, filter and compute psd on the raw data using *mne*.


## Libraries & Config

In [ ]:
import matplotlib
import pathlib
import mne
import numpy as np

# Set Matpllotlib backend to 'Qt5Agg' (best for MNE-Python's interactive plotting functions)
matplotlib.use('Qt5Agg')

## Retrieve data

Now we wanna work with some *sample* data and *mne* comes with bunch of datasets `mne.datasets` that we can use for analysis. Therefore, we will be using *mne* provided `mne.datsets.sample` audio/visual dataset which contains M\EEG and EOG recordings.

In [ ]:
# if the data is not avaible it will get downloaded automatically,
#  otherwise we will simply get pointed to the sample data dir.
sample_data_dir = mne.datasets.sample.data_path("./") 

# converting to a pathlib.Path for more convenience
sample_data_dir = pathlib.Path(sample_data_dir)
sample_data_dir

## Load/Read some raw data
To load or read way data we will be using the `mne.io.read_raw()` function.

In [ ]:
raw_data_path = sample_data_dir / 'MEG' / 'sample' / 'sample_audvis_raw.fif'

raw_data = mne.io.read_raw(raw_data_path)
raw_data

As we can see from the results above, the raw data we have loaded contains 5 channels in total, where the first 2 *Magnetometers* & *Gradiometers* represents the *MEG* channel with total 306 channels with 1 being a bad channel in the *Gradiometers* channel. There are 60 *EEG* channels with one bad channel. 1 *EOG* channel. The *Stimulus* channel represents the time-locked 'events' that took place during the recording.    

## Visualisation

To Visualise the raw deta, we can simply calls its `plot()` func.

In [ ]:
raw_data.plot()

# Event Extraction

To get all the info about the time-locked *STIM* events that took place during the recording of the raw data, we can use the `mne.find_events()` function.

In [ ]:
events = mne.find_events(raw_data)

events

We can see that there in total 6 discrete event with event ids `[ 1  2  3  4  5 32]` and combining all of them it is 320 total events recorded. The returned `events` object is numpy array with *3 columns* - The first column contains the event onset (in samples) with first_samp included. The last column contains the event code. The second column contains the signal value of the immediately preceding sample, and reflects the fact that event arrays sometimes originate from analog voltage channels (“trigger channels” or “stim channels”). In most cases, the second column is all zeros and can be ignored.

  Now to make the reading of the plots a bit easier we can label these event corresponding to their nature, and pass the `events` and `event_id` (telling which id refers to which label) to the `plot()` function.

In [ ]:
event_id = {
    'Auditory/Left' : 1,
    'Auditory/Right' : 2,
    'Visual/Left' : 3,
    'Visual/Right' : 4,
    'Smiley' : 5,
    'Button' : 32
} 
print(event_id)

# super-imposing
raw_data.plot(events=events, event_id=event_id)

Let's check how many button events and also visual events are in the data

In [ ]:
button_events = len(events[events[:, 2] == 32])
visual_events = len(events[np.isin(events[:, 2], [3, 4])])


print(f"Total button events are {button_events} & total visual events are {visual_events}")

## Gathering important info about the raw data

The info structure of any mne provided/compatible data is the most useful way to gather info about the data and we simply have to use the `.info()` functionality of the dat object.
It gives us the information about the *general parameters*, *acquisition settings*, *channels* and *filters*. 

In [ ]:
raw_data.info

Along with the info above, we can also actually call for other informations and all we need is the right key:

In [ ]:
for key, val in raw_data.info.items():
    print(key)

In [ ]:
raw_data.info['bads'] # list of bad recordings

we can also call the `.ch_names` to get all the channels name and to get further deep info about a particular channel we can call the `.info['chs'][ch_id]`

In [ ]:
print(raw_data.ch_names[:10]) # the first channel names

raw_data.info['chs'][0]

## Visualise the sensor locations
To visualise the locations of the sensors we can call the `.plot_sensors()` of the data object and pass the `ch_type=''` parameter to it. Furthermore, we can also get a 3D visualisation of the location by simply passing the param `kind=3d`.

> Note: The <code style="color : red">red bot</code> represents the bad channel.

In [ ]:
raw_data.plot_sensors(ch_type='eeg')

In [ ]:
raw_data.plot_sensors(kind='3d', ch_type='eeg')

## Mark channels as bad
To mark a channel as bad we simply has to add its name to the `raw.info['bads']` list. Let's try to add one of the EEG channels as a bad channel:

In [ ]:
import random

eeg_chs = [eeg_ch for eeg_ch in raw_data.ch_names if 'EEG ' in eeg_ch]

bad_eeg_ch = random.randint(0, len(eeg_chs)-1)
print(f"Marking the eeg channel '{eeg_chs[bad_eeg_ch]}' as bad")

raw_data.info['bads'] += [eeg_chs[bad_eeg_ch]] # make sure to put the eeg channel in a list
# visualise the sensor locations
raw_data.plot_sensors(ch_type='eeg')

## Select only a subset of the channels

As we know now that we have 204 *Gradiometers*, 102 *Magnetometers*, 60 *EEG*, 1 *EOG* & 9 *STIM* channels. Now, if we only want to look at a subset of these channels, for example, *EEG* and *EOG*, we have call the data object in-built function `pick()` and specify which sensor type to pick by passing the `picks=['eeg', 'eog']` param. So, 

In [ ]:
raw_eeg_eog = raw_data.copy().pick(
    picks=['eeg', 'eog'],
    exclude=[] # only imp to tell mne to not remove the bad channels, because normally it drops them
)

print(set([ch[:3] for ch in raw_eeg_eog.ch_names]))

In [ ]:
raw_eeg_eog.info

In [ ]:
raw_eeg_eog.plot(events=events, event_id=event_id)

Now let's only pick *MEG* channels and also one where we only pick the *magnetometer* channel ('MAG') of the *MEG* channels.

In [ ]:
raw_data.info

In [ ]:
raw_meg = raw_data.copy().pick(
    picks=['mag', 'grad'],
    exclude=[]
)

raw_meg.plot(events=events, event_id=event_id)

In [ ]:
raw_meg.info

In [ ]:
raw_mag = raw_meg.copy().pick(
    picks=['mag'],
    exclude=[]
)

raw_mag.info

In [ ]:
raw_mag.plot(events=events, event_id=event_id)

# Crop and Filter the data

Similar to `.pick()` in-built function to crop the data we just have to call the in-built `.crop()` function with the args `t_min` & `tmax`. Furthermore, we can also at the time-points of our data by calling `.times` attribute.

In [ ]:
# this line here will crop the data to a duration of 100 milliseconds
raw_eeg_eog_cropped = raw_eeg_eog.copy().crop(tmax=100)
# look at the latest time point in the cropped data
raw_eeg_eog_cropped.times[-1]

To filter the data we can call the `.filter()` and pass two of its many args `l_freq` and `h_freq`, these two params are the hertz bounds for frequencies (i.e., it will perform a band-pass filter with the given hertz cutoff).

| Low pass filter |	High pass filter |
|-----------------|------------------|
| It is used for smoothing the image. |	It is used for sharpening the image. |
| It attenuates the high frequency.	| It attenuates the low frequency. |
| Low frequency is preserved in it.	| High frequency is preserved in it. |
| It allows the frequencies below cut off frequency to pass through it.	It allows the | frequencies above cut off frequency to pass through it. |
| It consists of resistor that is followed by capacitor. | It consists of capacitor that is followed by a resistor. |
| It helps in removal of aliasing effect. | It helps in removal of noise. |
| G(u, v) = H(u, v) . F(u, v) | H(u, v) = 1 - H'(u, v) |

> Note: Since filtering needs all of the data in the memory, therefore we have to load the data using the in-built `.load_data()` function.

In [ ]:
raw_eeg_eog_cropped.load_data()
raw_eeg_eog_filtered = raw_eeg_eog_cropped.copy().filter(l_freq=0.1, h_freq=40)

In [ ]:
raw_eeg_eog.plot(events=events, event_id=event_id, title="raw_eeg_eog")
raw_eeg_eog_cropped.plot(events=events, event_id=event_id, title="cropped")
raw_eeg_eog_filtered.plot(events=events, event_id=event_id, title="filtered")

## PSD plotting

In time series analysis, a **PSD (Power Spectral Density)** is a measure of a signal's power distribution across different frequencies. It shows how much power is concentrated at each frequency, acting as a way to analyze a signal in the frequency domain rather than the time domain. *PSD* is often used for analyzing broadband random signals, and it can reveal periodicities and characteristics that are hidden in the raw time-domain data. 

To plot *PSD* of a data similar to the `.plot()` we call the `.plot_psd()`.

Normally, psd is usefull to check if the filtering has worked well or not.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(3)

raw_eeg_eog.compute_psd().plot(axes=ax[0], show=False)
raw_eeg_eog_cropped.compute_psd().plot(axes=ax[1], show=False)
raw_eeg_eog_filtered.compute_psd().plot(axes=ax[2], show=False)

ax[0].set_title('PSD on raw')
ax[1].set_title('PSD on cropped')
ax[2].set_title('PSD on filtered')
ax[2].set_xlabel('Frequency (Hz)')

fig.set_tight_layout(True)
plt.show()

Now, let's filter the raw data with a 1 Hz high-pass and a 30 Hz low-pass filter and plot the PSD.

In [ ]:
# crop the data
raw_data_cropped = raw_data.copy().crop(tmax=100)
raw_data_cropped.load_data()
raw_data_filtered = raw_data_cropped.copy().filter(l_freq=1, h_freq=30)

In [ ]:
raw_data.plot(events=events, event_id=event_id, title="raw_data")
raw_data_cropped.plot(events=events, event_id=event_id, title="raw_data_cropped")
raw_data_filtered.plot(events=events, event_id=event_id, title="raw_eeg_filtered")

In [ ]:
fig_2, ax_2 = plt.subplots(9)


raw_data.compute_psd().plot(axes=[ax_2[0], ax_2[1], ax_2[2]], show=False)
raw_data_cropped.compute_psd().plot(axes=[ax_2[3], ax_2[4], ax_2[5]], show=False)
raw_data_filtered.compute_psd().plot(axes=[ax_2[6], ax_2[7], ax_2[8]], show=False)

ax_2[0].set_title('PSD on raw')
ax_2[3].set_title('PSD on cropped')
ax_2[6].set_title('PSD on filtered')
ax_2[8].set_xlabel('Frequency (Hz)')

fig_2.set_tight_layout(True)
plt.show()

## Saving the data

To save the filtered data we simply have to call its `.save()` function.

In [ ]:
from pathlib import Path

output_dir = 'output_data/ch_1'
Path(output_dir).mkdir(parents=True, exist_ok=True)

raw_eeg_eog_filtered.save(
    Path(output_dir) / 'eeg_eog_cropped_filt_raw.fif',
    overwrite=True
)